In [18]:
from pypot.feetech import FeetechSTS3215IO
import time

## Motor initialisation

In [19]:
# Waveshare bus servo adapter (CH343 chip, USB 1a86:55d3) on this Mac.
# If it changes, find it with: ls /dev/cu.usbmodem*
PORT = '/dev/cu.usbmodem58CD1773071'
BAUDRATE = 1000000  # STS3215 factory default

gripper = FeetechSTS3215IO(port=PORT, baudrate=BAUDRATE)

Set the ID of the motor, that should be configured in its firmware before the assembly.

In [20]:
ID = 7  # this motor's current ID (the README / SO-ARM100 convention is 6)

assert gripper.ping(ID), f"No motor answering on ID {ID}: check 12V power, servo cable and the adapter jumper"
print("Motor", ID, "found, position:", gripper.get_present_position([ID]))

Motor 7 found, position: (22.37,)


Try if the motor get stiff

In [21]:
gripper.enable_torque([ID])

Then set it compliant again.

In [22]:
gripper.disable_torque([ID])

## Flash the motors usefull parameters

First, open the gripper and check that the value is close to -140° and not a value completely elsewhere

In [37]:
gripper.get_present_position([ID])

(-142.2,)

Then do it again, closed this time, and check the value is around 0°

In [33]:
gripper.get_present_position([ID])

(0.22,)

If everything went well, then we can flash the needed parameters:

> ⚠️ This writes permanent limits to the motor's EEPROM (angle range [-147°, 0°]). Only run it if the open/closed readings above were close to -140° and 0°. If they are far off, re-mount the motor horn first.

In [38]:
gripper.set_lock({ID: 0})
time.sleep(0.5)
gripper.set_acceleration({ID: 200})
time.sleep(0.5)

gripper.set_max_angle_limit({ID: 0})
time.sleep(0.5)
gripper.set_min_angle_limit({ID: -147})
time.sleep(0.5)

gripper.set_torque_limit({ID: 1000})
time.sleep(0.5)

gripper.set_overload_torque({ID: 40})
time.sleep(0.5)

gripper.set_protective_torque({ID: 5})
time.sleep(0.5)

gripper.set_protection_time({ID: 7})
time.sleep(0.5)

gripper.set_lock({ID: 1})

Remarks : 
- To change the general torque limit: Register 16 set_torque_limit and not max_torque_limit.
- For fast movement, we must not limit the current, which proportionally decreases the speed.
- So we set an 'overload torque' threshold at 65%, which engages the protective torque set at 20%.
- To ensure reactivity, we set the protection time to 2, i.e. 20ms. 10ms sometimes too sensitive

## Try the gripper!  
Let's define simple methods to use it easily

In [39]:
def open():
    gripper.enable_torque([ID])
    gripper.set_goal_position({ID: -140})

def close():
    gripper.enable_torque([ID])
    gripper.set_goal_position({ID: 0})

Then try them:

In [44]:
open()

In [45]:
close()